# parte 4 - aprendizagem nao supervisionada
## 4.c analise de associacao (apriori) e 4.d deteccao de outliers (lof)

neste notebook vamos usar o apriori para encontrar regras de associacao
entre as caracteristicas das casas e o lof para identificar outliers.

In [ ]:
# importando as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_rows', 200)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# carregando os dados
df = pd.read_csv('train.csv')

## repetindo as transformacoes que o felipe fez

In [ ]:
# target encoding no bairro
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['Neighborhood'])
df['Neighborhood'] = encoder.fit_transform(df['Neighborhood'], df['SalePrice'])

In [ ]:
# transformando variaveis categoricas em numeros
colunas_ordinais = ['ExterQual', 'KitchenQual', 'BsmtQual']
mapeamento = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1}

df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento)

df['BsmtQual'] = df['BsmtQual'].fillna('Po')
mapeamento_completo = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento_completo)

In [ ]:
# criando as features novas
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
df['TotalArea'] = df['GrLivArea'] + df['TotalBsmtSF']
df['TotalBath'] = (df['FullBath'] + 0.5*df['HalfBath']
                   + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)

In [ ]:
# selecionando as mesmas features
features = [
    'OverallQual', 'TotalArea', 'Neighborhood', 'ExterQual',
    'KitchenQual', 'GarageCars', 'TotalBath', 'BsmtQual',
    'HouseAge', 'YearsSinceRemodel'
]

X = df[features].fillna(0)

# padronizando
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)